# The bright-star mask — field 860

The pipeline masks every object inside a Gaia G < 16 star's circle, R = 1630″ × 1.396^(−G) (Legacy Surveys DR9), or inside
one of its four diffraction-spike boxes (PA 0, 90, 30, 150; 12″ wide; half-length R × 1.15^(16 − G)). Both come from
`Source_Analysis/filter_field.py`; the spikes were tuned by eye on these images.

In [ ]:
import os, sys, io, warnings; sys.path.insert(0, '..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from collections import OrderedDict
from IPython.display import display
from scipy.spatial import cKDTree
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D, NoOverlapError
from astropy.coordinates import SkyCoord
from astropy.visualization import simple_norm
import astropy.units as u
from matplotlib.patches import Circle
from Extracting.images import load_refimg
from Extracting.ztf_products import refimg_path
from Source_Analysis import filter_field as ff
from source_db.cutouts.render import DEFAULT_STRETCH, DEFAULT_PERCENT, ps1_cutout_url, ps1_linear_flux, _get_fits
from source_db.cutouts.panstarrs import _post as ps1_filenames
warnings.filterwarnings('ignore')
FIELD, D = 860, '../Data'
REVIEW = f'{D}/fields/{FIELD:06d}/review/star_mask'; os.makedirs(REVIEW, exist_ok=True)
stars = ff.bright_stars(FIELD, D)
ztf = pd.read_parquet(f'{D}/fields/{FIELD:06d}/ztf_r.parquet', columns=['ra', 'dec', 'ccd', 'qid'])
ps1 = pd.read_parquet(f'{D}/fields/{FIELD:06d}/ps1.parquet', columns=['ra', 'dec'])
cosd = np.cos(np.radians(stars.dec.median()))
trees = {'ZTF': cKDTree(np.c_[ztf.ra.values * cosd, ztf.dec.values]), 'PS1': cKDTree(np.c_[ps1.ra.values * cosd, ps1.dec.values])}
_, j = trees['ZTF'].query(np.c_[stars.ra.values * cosd, stars.dec.values]); stars['ccd'], stars['qid'] = ztf.ccd.values[j], ztf.qid.values[j]
G_grid = np.arange(4, 16.01, 1.0)
display(pd.DataFrame({'G': G_grid, 'R ["]': ff.bright_star_radius(G_grid).round(0), 'spike half-length ["]': ff.spike_half_length(G_grid).round(0)}).set_index('G').T)

CasJobs will use the credentials from mast_dino_login.txt
The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


G,4.0,5.0,6.0,7.0,8.0,9.0,10.0,11.0,12.0,13.0,14.0,15.0,16.0
"R [""]",429.0,307.0,220.0,158.0,113.0,81.0,58.0,42.0,30.0,21.0,15.0,11.0,8.0
"spike half-length [""]",2296.0,1430.0,891.0,555.0,346.0,215.0,134.0,84.0,52.0,32.0,20.0,13.0,8.0


## 1. The mask on the images

Three stars per G bin: ZTF r reference over PS1 r stack, north up, wide enough for the whole mask. Red: the circle;
cyan dashed: the spike boxes (ZTF only — PS1 stacks rotate the spikes away). White dots: every ZTF and PS1 detection.
Saved to `review/star_mask/`.

In [ ]:
G_BINS = [(4, 8), (8, 10), (10, 12), (12, 14), (14, 16)]
N_PER = 3
rng = np.random.default_rng(0)
_refs = OrderedDict()
def ref_image(ccd, qid, b='r'):
    key = (ccd, qid, b)
    if key not in _refs:
        if len(_refs) >= 8: _refs.popitem(last=False)
        _refs[key] = load_refimg(refimg_path(f'{D}/ztf_data', FIELD, *key))
    return _refs[key]
def ps1_stamp(ra, dec, size, files):
    name = files.get((round(ra, 6), round(dec, 6), 'r'))
    raw = _get_fits(ps1_cutout_url(ra, dec, name, int(min(size / 0.25, 6000)))) if name else None
    if not raw: return None, None
    hdu = fits.open(io.BytesIO(raw))[0]
    return ps1_linear_flux(hdu.data.astype(np.float32), hdu.header), WCS(hdu.header)
def draw_mask(ax, x0, y0, G, scale, boxes=True):
    ax.add_patch(Circle((x0, y0), ff.bright_star_radius(G) / scale, fill=False, color='red', lw=1.6))
    if not boxes: return
    L = float(ff.spike_half_length(G))
    for pa in ff.SPIKE_PA:
        th = np.radians(pa); ex, ey = -np.sin(th), np.cos(th)
        ux, uy = ex * L / scale, ey * L / scale; wx, wy = -ey * ff.SPIKE_WIDTH_ARCSEC / 2 / scale, ex * ff.SPIKE_WIDTH_ARCSEC / 2 / scale
        ax.add_patch(plt.Polygon([[x0 + ux + wx, y0 + uy + wy], [x0 + ux - wx, y0 + uy - wy], [x0 - ux - wx, y0 - uy - wy], [x0 - ux + wx, y0 - uy + wy]], fill=False, color='cyan', lw=1.2, ls='--'))
def draw(ax, data, wcs, star, size, survey):
    ax.set_xticks([]); ax.set_yticks([])
    if data is None or not np.isfinite(data).any():
        ax.text(0.5, 0.5, 'no image', ha='center', va='center', transform=ax.transAxes); return
    flip = wcs.pixel_scale_matrix[0, 0] > 0; data = data[::-1, ::-1] if flip else data
    scale = abs(wcs.pixel_scale_matrix[1, 1]) * 3600
    def pix(ra, dec):
        x, y = wcs.world_to_pixel_values(np.asarray(ra), np.asarray(dec))
        return (data.shape[1] - 1 - x, data.shape[0] - 1 - y) if flip else (x, y)
    ax.imshow(data, origin='lower', cmap='viridis', norm=simple_norm(data, DEFAULT_STRETCH, percent=DEFAULT_PERCENT))
    for k in ('ZTF', 'PS1'):
        cat = ztf if k == 'ZTF' else ps1
        hit = cat.iloc[trees[k].query_ball_point([star.ra * cosd, star.dec], size * 0.75 / 3600)]
        x, y = pix(hit.ra.values, hit.dec.values); ax.scatter(x, y, s=6, color='white', alpha=0.5, lw=0)
    x0, y0 = pix([star.ra], [star.dec]); draw_mask(ax, float(x0[0]), float(y0[0]), star.phot_g_mean_mag, scale, boxes=(survey == 'ZTF'))
    ax.plot(x0, y0, '+', color='red', ms=10, mew=1.2)
    ax.set_xlim(-0.5, data.shape[1] - 0.5); ax.set_ylim(-0.5, data.shape[0] - 0.5)
    bar = [b_ for b_ in (5, 10, 20, 30, 60, 120, 300) if b_ <= size / 4][-1]; x1, y1 = 0.05 * data.shape[1], 0.06 * data.shape[0]
    ax.plot([x1, x1 + bar / scale], [y1, y1], color='white', lw=2.5); ax.text(x1 + bar / scale / 2, y1 + 0.02 * data.shape[0], f'{bar}″', color='white', fontsize=8, ha='center', va='bottom')
    ax.text(0.03, 0.97, f'{survey} r', color='white', fontsize=9, ha='left', va='top', transform=ax.transAxes)
for lo, hi in G_BINS:
    sample = stars[(stars.phot_g_mean_mag >= lo) & (stars.phot_g_mean_mag < hi)].pipe(lambda t: t.sample(min(N_PER, len(t)), random_state=int(rng.integers(1 << 30))).sort_values('phot_g_mean_mag'))
    files = ps1_filenames(sample[['ra', 'dec']]).round({'ra': 6, 'dec': 6}).groupby(['ra', 'dec', 'filter']).filename.first()
    fig, axes = plt.subplots(2, len(sample), figsize=(4.4 * len(sample), 9), squeeze=False)
    for j, (_, st) in enumerate(sample.iterrows()):
        size = max(2.3 * max(ff.bright_star_radius(st.phot_g_mean_mag), float(ff.spike_half_length(st.phot_g_mean_mag))), 60)
        try:
            im = ref_image(int(st.ccd), int(st.qid))
            cut = Cutout2D(im.data, SkyCoord(st.ra * u.deg, st.dec * u.deg), size * u.arcsec, wcs=im.wcs, mode='partial'); draw(axes[0, j], cut.data, cut.wcs, st, size, 'ZTF')
        except NoOverlapError: draw(axes[0, j], None, None, st, size, 'ZTF')
        draw(axes[1, j], *ps1_stamp(st.ra, st.dec, size, files), st, size, 'PS1')
        axes[0, j].set_title(f'G = {st.phot_g_mean_mag:.2f}   R {ff.bright_star_radius(st.phot_g_mean_mag):.0f}″  spikes {float(ff.spike_half_length(st.phot_g_mean_mag)):.0f}″   ({size:.0f}″ wide)', fontsize=9)
        axes[1, j].set_xlabel(f'{st.ra:.4f} {st.dec:+.4f}', fontsize=8)
    out = f'{REVIEW}/G_{lo:02d}-{hi:02d}.png'; plt.tight_layout(); fig.savefig(out, dpi=80); plt.close(fig); print(os.path.abspath(out))

/Users/adamboesky/Research/long_transients2/Data/fields/000860/review/star_mask/G_04-08.png
/Users/adamboesky/Research/long_transients2/Data/fields/000860/review/star_mask/G_08-10.png
/Users/adamboesky/Research/long_transients2/Data/fields/000860/review/star_mask/G_10-12.png
/Users/adamboesky/Research/long_transients2/Data/fields/000860/review/star_mask/G_12-14.png
/Users/adamboesky/Research/long_transients2/Data/fields/000860/review/star_mask/G_14-16.png


## 2. Detection overdensity around the stars

Surface density of ZTF r detections and PS1 objects vs distance from the star, stacked per G bin, relative to the field
mean (measured 100–300″ from the G ≥ 14 stars). Red dashed: R at the bin centre. The dip at 20–35″ around fainter stars
is where DAOPhot misses sources next to a star.

In [ ]:
r_edges = np.geomspace(2, 300, 30); r_mid = np.sqrt(r_edges[1:] * r_edges[:-1]); area = np.pi * (r_edges[1:] ** 2 - r_edges[:-1] ** 2)
cats = {'ZTF r detections': ztf, 'PS1 objects': ps1}
def stacked_counts(k, sel):
    cat = cats[k]; xy = np.c_[cat.ra.values * cosd, cat.dec.values]; counts = np.zeros(len(r_mid))
    for ra, dec, hits in zip(sel.ra.values, sel.dec.values, trees[k].query_ball_point(np.c_[sel.ra.values * cosd, sel.dec.values], r_edges[-1] / 3600)):
        if hits: counts += np.histogram(np.hypot(xy[hits, 0] - ra * cosd, xy[hits, 1] - dec) * 3600, r_edges)[0]
    return counts
calib = stars[stars.phot_g_mean_mag >= 14]; outer = r_mid >= 100
mean = {k: stacked_counts(k, calib)[outer].sum() / (len(calib) * area[outer].sum()) for k in cats}
PROFILE_BINS = [(4, 8), (8, 10), (10, 11), (11, 12), (12, 13), (13, 14), (14, 15), (15, 16)]
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharex=True, sharey=True); axes = axes.ravel(); profiles = {}
for ax, (lo, hi) in zip(axes, PROFILE_BINS):
    sel = stars[(stars.phot_g_mean_mag >= lo) & (stars.phot_g_mean_mag < hi)]
    for k, col in zip(cats, ('C0', 'C1')):
        profiles[(lo, hi, k)] = stacked_counts(k, sel) / (len(sel) * area * mean[k]); ax.plot(r_mid, profiles[(lo, hi, k)], color=col, lw=1.6, label=k)
    ax.axhspan(0.95, 1.05, color='green', alpha=0.08); ax.axvline(ff.bright_star_radius(0.5 * (lo + hi)), color='red', ls='--', lw=1.2, label='R (bin centre)')
    ax.axhline(1, color='k', lw=0.6, ls=':'); ax.set_xscale('log'); ax.set_yscale('log'); ax.set_ylim(0.1, 300); ax.set_xlim(2, 300); ax.set_title(f'G {lo}–{hi}  ({len(sel):,} stars)')
    if ax is axes[0]: ax.legend(frameon=False, fontsize=8)
for ax in axes[4:]: ax.set_xlabel('distance from the star [arcsec]')
axes[0].set_ylabel('density / field mean'); axes[4].set_ylabel('density / field mean'); plt.tight_layout(); plt.show()

KeyError: 'ZTF r detections'

## 3. Measured radius vs G

Per bin, the radius beyond which the ZTF density stays within ±5% of the field mean (the first bin after the outermost
one outside the band), against the pipeline's R.

In [ ]:
def flat_radius(ratio, tol=0.05):
    hits = np.flatnonzero(np.abs(ratio - 1) > tol)
    return r_mid[0] if not len(hits) else (r_mid[hits[-1] + 1] if hits[-1] + 1 < len(r_mid) else np.nan)
pts = np.array([(0.5 * (lo + hi), lo, hi, flat_radius(profiles[(lo, hi, 'ZTF r detections')])) for lo, hi in PROFILE_BINS])
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(pts[:, 0], pts[:, 3], xerr=[pts[:, 0] - pts[:, 1], pts[:, 2] - pts[:, 0]], fmt='o', color='green', ms=7, capsize=3, label='flat beyond (within ±5% of the mean)')
G = np.linspace(4, 16, 200); ax.plot(G, ff.bright_star_radius(G), color='red', lw=1.6, label='pipeline R = 1630″ × 1.396$^{-G}$'); ax.plot(G, ff.spike_half_length(G), color='cyan', lw=1.6, ls='--', label='spike half-length')
ax.set_yscale('log'); ax.set_xlabel('Gaia G'); ax.set_ylabel('radius [arcsec]'); ax.set_xlim(4, 16); ax.legend(frameon=False); ax.grid(alpha=0.3, which='both'); plt.tight_layout(); plt.show()

NameError: name 'PROFILE_BINS' is not defined